# **|| AI DevOps Log Analyzer |**|

An AI-powered log analysis system that reads server/system logs, detects anomalies, identifies root causes and generates a structured incident report using a RAG (Retrieval Augmented Generation) pipeline.

**What it does:**
- Parses and preprocesses raw log files
- Detects anomalies (errors, warnings, critical events)
- Uses RAG to retrieve relevant log context before querying the LLM
- Suggests root causes for detected issues
- Generates a structured incident report

**Tech used:** `Groq` · `ChromaDB` · `sentence-transformers` · `LangChain`


## **1** - ***Installation***

In [ ]:
%%capture
!pip install groq chromadb sentence-transformers langchain langchain-community rich

## **2** - ***Imports and Setting***s

In [ ]:
import os
import re
import json
import time
from datetime import datetime
from getpass import getpass

from groq import Groq
import chromadb
from sentence_transformers import SentenceTransformer
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.markdown import Markdown

os.environ["PYTHONIOENCODING"] = "utf-8"

# --- Settings ---
MODEL_NAME     = "llama-3.1-8b-instant"   # Groq model
EMBED_MODEL    = "all-MiniLM-L6-v2"       # lightweight embedding model for RAG
CHUNK_SIZE     = 10                       # Number of log lines per chunk
TOP_K          = 5                        # Number of relevant chunks to retrieve for each query
MAX_TOKENS     = 1024
TEMPERATURE    = 0.3

# Log levels we consider as anomalies
ANOMALY_LEVELS = ["ERROR", "CRITICAL", "FATAL", "WARNING", "WARN", "EXCEPTION", "TRACEBACK"]

console = Console()
print("Imports done!")

## **3** - ***API Key Setup***

> Free Groq API key at https://console.groq.com

In [ ]:
GROQ_API_KEY = getpass("Enter your Groq API key: ")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

client = Groq(api_key=GROQ_API_KEY)
print("API key saved!")

## **4** - ***Load Logs***

You can either upload a log file or paste log content directly. Run whichever section fits your use case.



### **Option A — Upload a log file**

In [ ]:
from google.colab import files

print("Upload your log file (.log or .txt):")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    with open(filename, "r", errors="ignore") as f:
        raw_logs = f.read()
    print(f"Loaded {len(raw_logs.splitlines())} lines from {filename}")
else:
    print("No file uploaded. Use Option B below to paste logs directly.")

### **Option B — Paste logs directly**

If you don't have a log file, a sample log is included below so you can still run the full pipeline.

In [ ]:
# Replace the content below with your own logs, or just run as-is with the sample.
raw_logs = """
2024-01-15 08:00:01 INFO - Server started successfully on port 8080
2024-01-15 08:01:12 INFO - Database connection established
2024-01-15 08:05:34 INFO - User authentication service initialized
2024-01-15 08:12:45 WARNING - High memory usage detected: 87% of available RAM in use
2024-01-15 08:13:01 INFO - Cache cleared successfully
2024-01-15 08:15:22 ERROR - Database connection timeout after 30s — retrying (attempt 1/3)
2024-01-15 08:15:53 ERROR - Database connection timeout after 30s — retrying (attempt 2/3)
2024-01-15 08:16:24 ERROR - Database connection timeout after 30s — retrying (attempt 3/3)
2024-01-15 08:16:25 CRITICAL - Database connection failed — all retry attempts exhausted
2024-01-15 08:16:26 ERROR - Unable to process user requests — database unavailable
2024-01-15 08:17:00 WARNING - Falling back to read-only cache mode
2024-01-15 08:18:10 ERROR - Payment service timeout: gateway did not respond within 10s
2024-01-15 08:19:45 INFO - Health check ping received
2024-01-15 08:20:00 ERROR - API rate limit exceeded for service: auth-provider
2024-01-15 08:21:30 WARNING - Disk usage at 91% on /var/log partition
2024-01-15 08:22:05 CRITICAL - Out of memory — killing process 4821 (worker)
2024-01-15 08:22:06 ERROR - Worker process crashed unexpectedly — restarting
2024-01-15 08:23:00 INFO - Worker process restarted successfully
2024-01-15 08:25:10 WARNING - Response time degraded — avg latency 4200ms (threshold: 2000ms)
2024-01-15 08:26:00 ERROR - SSL certificate validation failed for external-api.example.com
2024-01-15 08:27:15 INFO - Scheduled backup started
2024-01-15 08:28:00 ERROR - Backup failed — insufficient disk space
2024-01-15 08:30:00 INFO - System status check completed
""".strip()

print(f"Loaded {len(raw_logs.splitlines())} log lines")
print("\nFirst few lines:")
for line in raw_logs.splitlines()[:5]:
    print(" ", line)

## **5** - ***Log Parsing and Preprocessing***

In [ ]:
def parse_log_line(line):
    """
    Tries to parse a log line into its components: timestamp, level, message.
    Returns a dict. If parsing fails, stores the raw line as the message.
    """
    # Common log format: YYYY-MM-DD HH:MM:SS LEVEL message
    pattern = r"(\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}:\d{2})\s+(\w+)\s+(.*)"
    match = re.match(pattern, line.strip())

    if match:
        return {
            "timestamp": match.group(1),
            "level":     match.group(2).upper(),
            "message":   match.group(3),
            "raw":       line.strip()
        }
    else:
        # Line doesn't match expected format, store as is.
        return {
            "timestamp": "unknown",
            "level":     "UNKNOWN",
            "message":   line.strip(),
            "raw":       line.strip()
        }


# Parse all lines
parsed_logs = []
for line in raw_logs.splitlines():
    if line.strip():  # Skip empty lines
        parsed_logs.append(parse_log_line(line))

print(f"Parsed {len(parsed_logs)} log entries")

# Count by level
level_counts = {}
for log in parsed_logs:
    level = log["level"]
    level_counts[level] = level_counts.get(level, 0) + 1

print("\nLog level breakdown:")
for level, count in sorted(level_counts.items(), key=lambda x: -x[1]):
    print(f"  {level}: {count}")

## **6** - ***Anomaly Detection***

In [ ]:
def detect_anomalies(parsed_logs):
    """
    Scans parsed logs and flags entries that are errors, warnings or critical events.
    Returns a list of anomalous log entries.
    """
    anomalies = []
    for log in parsed_logs:
        if log["level"] in ANOMALY_LEVELS:
            anomalies.append(log)
    return anomalies


anomalies = detect_anomalies(parsed_logs)
print(f"Found {len(anomalies)} anomalies out of {len(parsed_logs)} total log entries")

# Show them in a table
table = Table(title="Detected Anomalies", show_lines=True)
table.add_column("Timestamp", style="dim")
table.add_column("Level", style="bold red")
table.add_column("Message")

for a in anomalies:
    color = "red" if a["level"] in ["ERROR", "CRITICAL", "FATAL"] else "yellow"
    table.add_row(
        a["timestamp"],
        f"[{color}]{a['level']}[/{color}]",
        a["message"][:80]
    )

console.print(table)

## **7** - ***Build RAG Knowledge Base***

We chunk the logs, embed them using a sentence transformer and store them in ChromaDB. This lets us retrieve the most relevant log chunks for any query instead of dumping everything into the LLM prompt.

In [ ]:
def chunk_logs(parsed_logs, chunk_size=CHUNK_SIZE):
    """
    Groups log lines into chunks of chunk_size lines each.
    Each chunk is stored as a single string for embedding.
    """
    chunks = []
    for i in range(0, len(parsed_logs), chunk_size):
        batch = parsed_logs[i:i + chunk_size]
        chunk_text = "\n".join([log["raw"] for log in batch])
        chunks.append({
            "id":   f"chunk_{i}",
            "text": chunk_text,
            "start_line": i,
            "end_line":   i + len(batch)
        })
    return chunks


print("Loading embedding model...")
# all-MiniLM-L6-v2 is a small but good embedding model. No API key needed.
embedder = SentenceTransformer(EMBED_MODEL)
print("Embedding model loaded!")

# Chunk the logs
chunks = chunk_logs(parsed_logs)
print(f"Created {len(chunks)} chunks from {len(parsed_logs)} log lines")

# Set up ChromaDB in memory
chroma_client = chromadb.Client()

# Delete collection if it already exists (useful when re-running the cell)
try:
    chroma_client.delete_collection("logs")
except:
    pass

collection = chroma_client.create_collection("logs")

# Embed each chunk and store in ChromaDB
print("Embedding and indexing log chunks...")
texts = [chunk["text"] for chunk in chunks]
ids   = [chunk["id"]   for chunk in chunks]
embeddings = embedder.encode(texts).tolist()

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=ids
)

print(f"RAG knowledge base ready with {len(chunks)} indexed chunks!")

## **8**- ***RAG Retrieval and Root Cause Analysis***

In [ ]:
def retrieve_relevant_chunks(query, top_k=TOP_K):
    """
    Embeds the query and retrieves the most relevant log chunks from ChromaDB.
    This is the retrieval part of RAG.
    """
    query_embedding = embedder.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=min(top_k, len(chunks))  # Can't retrieve more than we have
    )
    return results["documents"][0]  # List of relevant chunk texts


def call_llm(system_prompt, user_prompt):
    """Calls the Groq LLM and returns the response text"""
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt}
            ],
            max_tokens=MAX_TOKENS,
            temperature=TEMPERATURE
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"LLM error: {e}. Retrying in 5 seconds...")
        time.sleep(5)
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt}
            ],
            max_tokens=MAX_TOKENS,
            temperature=TEMPERATURE
        )
        return response.choices[0].message.content.strip()


def analyze_root_cause(anomalies):
    """
    For each anomaly, retrieves relevant log context using RAG
    and asks the LLM to suggest the root cause.
    """
    system_prompt = """
        You are a senior DevOps engineer analyzing server logs.
        You will be given a log anomaly and relevant surrounding log context.
        Your job is to suggest the most likely root cause in 2-3 sentences.
        Be specific and technical. Only use the log context provided.
    """

    root_causes = []

    # Group anomalies by type to avoid too many LLM calls
    # We analyze unique error messages rather than every single line
    seen = set()
    unique_anomalies = []
    for a in anomalies:
        key = a["message"][:50]  # Use first 50 chars as dedup key
        if key not in seen:
            seen.add(key)
            unique_anomalies.append(a)

    print(f"Analyzing {len(unique_anomalies)} unique anomalies...")

    for anomaly in unique_anomalies:
        # RAG: retrieve relevant log chunks for this specific anomaly
        query = f"{anomaly['level']} {anomaly['message']}"
        relevant_chunks = retrieve_relevant_chunks(query)
        context = "\n".join(relevant_chunks)

        user_prompt = f"""Anomaly detected:
Timestamp: {anomaly['timestamp']}
Level: {anomaly['level']}
Message: {anomaly['message']}

Relevant log context (retrieved from full log):
{context}

What is the most likely root cause of this anomaly?"""

        root_cause = call_llm(system_prompt, user_prompt)
        root_causes.append({
            "anomaly": anomaly,
            "root_cause": root_cause
        })

        time.sleep(0.5)  # Small pause between calls

    return root_causes


print("Running root cause analysis...")
root_cause_results = analyze_root_cause(anomalies)

print("\nRoot Cause Analysis Results:")
print("=" * 60)
for item in root_cause_results:
    print(f"\n[{item['anomaly']['level']}] {item['anomaly']['message']}")
    print(f"Root Cause: {item['root_cause']}")
    print("-" * 60)

## **9** - ***Incident Report Generation***

In [ ]:
def generate_incident_report(anomalies, root_cause_results, parsed_logs):
    """
    Uses the LLM to generate a structured incident report
    based on all detected anomalies and their root causes.
    """
    system_prompt = """
        You are a DevOps engineer writing a formal incident report.
        Based on the anomalies and root causes provided, write a structured report in Markdown.

        The report must include:
        1. ## Incident Summary
        2. ## Timeline of Events
        3. ## Detected Anomalies
        4. ## Root Cause Analysis
        5. ## Recommended Actions
        6. ## Severity Assessment (Low / Medium / High / Critical)

        Be concise and technical. Only use information provided to you.
    """

    # Build a summary of everything for the LLM
    anomaly_summary = ""
    for item in root_cause_results:
        a = item["anomaly"]
        anomaly_summary += f"- [{a['level']}] {a['timestamp']}: {a['message']}\n"
        anomaly_summary += f"  Root cause: {item['root_cause']}\n\n"

    user_prompt = f"""Log analysis results:

Total log entries analyzed: {len(parsed_logs)}
Total anomalies detected: {len(anomalies)}
Log time range: {parsed_logs[0]['timestamp']} to {parsed_logs[-1]['timestamp']}

Anomalies and root causes:
{anomaly_summary}

Generate a complete incident report based on this information."""

    report = call_llm(system_prompt, user_prompt)
    return report


print("Generating incident report...")
incident_report = generate_incident_report(anomalies, root_cause_results, parsed_logs)

console.print(Panel("[bright_cyan]Incident Report[/bright_cyan]", expand=False))
console.print(Markdown(incident_report))

## **10** - ***Save the Report***

In [ ]:
def save_report(incident_report, root_cause_results):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Save markdown report
    md_filename = f"incident_report_{timestamp}.md"
    with open(md_filename, "w") as f:
        f.write(f"# Incident Report\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        f.write(incident_report)

    # Save raw root cause analysis as json
    json_filename = f"root_cause_analysis_{timestamp}.json"
    data = [
        {
            "timestamp":  item["anomaly"]["timestamp"],
            "level":      item["anomaly"]["level"],
            "message":    item["anomaly"]["message"],
            "root_cause": item["root_cause"]
        }
        for item in root_cause_results
    ]
    with open(json_filename, "w") as f:
        json.dump(data, f, indent=2)

    print(f"Saved: {md_filename}")
    print(f"Saved: {json_filename}")
    return md_filename, json_filename


md_file, json_file = save_report(incident_report, root_cause_results)

# Auto download in Colab
try:
    from google.colab import files
    files.download(md_file)
    files.download(json_file)
    print("Download started!")
except ImportError:
    print("Files saved in current directory.")

## **11** - ***Ask a Question About Your Logs***

You can also query the RAG system directly with any question about your logs.

In [ ]:
def ask_logs(question):
    """
    Ask any question about your logs.
    RAG retrieves the relevant context, LLM answers based on it.
    """
    system_prompt = """
        You are a DevOps assistant. Answer questions about server logs.
        Only use the log context provided to you. Be concise and specific.
    """

    relevant_chunks = retrieve_relevant_chunks(question)
    context = "\n".join(relevant_chunks)

    user_prompt = f"""Question: {question}

Relevant log context:
{context}

Answer based only on the log context above."""

    answer = call_llm(system_prompt, user_prompt)
    print(f"Q: {question}")
    print(f"A: {answer}")


# Example questions: Change these to whatever you want to ask
ask_logs("What caused the database connection failure?")
print()
ask_logs("Were there any memory related issues?")

## ****Things You Can Change!***

- `CHUNK_SIZE` — smaller chunks give more precise retrieval, larger chunks give more context per result
- `TOP_K` — how many chunks to retrieve per query, increase for more context
- `ANOMALY_LEVELS` — add or remove log levels to change what counts as an anomaly
- `EMBED_MODEL` — swap to a larger sentence transformer for better retrieval quality
- The questions in the last cell — ask anything about your logs
